In [1]:
import pandas as pd
import os
import requests
import datetime

import sys
sys.path.append('..')
from helpers import get_dst_timestamps

In [2]:
def convert_to_utc_timestamp(hour_ending, dst_start, dst_end):
    if '24:00' in hour_ending:
        timestamp = (
            pd.to_datetime(hour_ending.replace('24:00', '00:00')) + pd.Timedelta(days=1)
        )
    elif 'DST' in hour_ending:
        timestamp = pd.to_datetime(hour_ending.replace(' DST', ''))
    else:
        timestamp = pd.to_datetime(hour_ending)

    if (timestamp > dst_start) and (timestamp <= dst_end) and ('DST' not in hour_ending):
        timestamp -= pd.Timedelta(hours=1)

    timestamp += pd.Timedelta(hours=6)

    return timestamp

In [3]:
ercot_weather_zone_subba_map = {
    'COAST': 'COAS',
    'EAST': 'EAST',
    'FWEST': 'FWES',
    'FAR_WEST': 'FWES',
    'NORTH': 'NRTH',
    'NCENT': 'NCEN',
    'NORTH_C': 'NCEN',
    'SOUTH': 'SOUT',
    'SOUTHERN': 'SOUT',
    'WEST': 'WEST',
    'SCENT': 'SCEN',
    'SOUTH_C': 'SCEN'
}

In [57]:
# Profiles downloaded from https://www.ercot.com/gridinfo/load/load_hist
load_profile_list = []
dir = '../data/iso_load_profiles/raw/ercot'
for filename in os.listdir(dir):
    f = os.path.join(dir, filename)
    df = pd.read_excel(f)

    hour_end_column = df.columns[0]
    year = int(filename.split('.')[0].rsplit('_', maxsplit=1)[1])
    dst_start, dst_end = get_dst_timestamps(year)
    
    try:
        df['timestamp'] = df[hour_end_column].apply(lambda x: x.round('h'))
        df.loc[(df.timestamp > dst_start) & (df.timestamp <= dst_end), 'timestamp'] -= (
            pd.Timedelta(hours=1)
        )
        df.loc[df.timestamp.duplicated(keep='last'), 'timestamp'] = dst_end
        df['timestamp'] += pd.Timedelta(hours=6)
    except:
        df['timestamp'] = (
            df[hour_end_column]
            .astype(str)
            .apply(convert_to_utc_timestamp, dst_start=dst_start, dst_end=dst_end)
            .apply(lambda x: x.round('h'))
        )

    assert len(df.drop_duplicates('timestamp')) == len(df)

    df = df.drop(columns=[hour_end_column, 'ERCOT'])
    df = pd.melt(
        df,
        id_vars='timestamp',
        value_vars=df.columns,
        var_name='weather_zone'
    )
    df['subba'] = df['weather_zone'].map(ercot_weather_zone_subba_map)
    df = df.drop(columns='weather_zone')

    load_profile_list.append(df)

In [ ]:
ercot_load = pd.concat(load_profile_list, ignore_index=True)
ercot_load.to_csv(f"../data/iso_load_profiles/ercot.csv", index=False)

In [3]:
ercot_weather_zone_subba_map = {
    'Coast': 'COAS',
    'East': 'EAST',
    'FarWest': 'FWES',
    'North': 'NRTH',
    'NorthCentral': 'NCEN',
    'SouthCentral': 'SCEN',
    'Southern': 'SOUT',
    'West': 'WEST'
}

In [4]:
forecast_profile_list_temp = []
dir = '../data/iso_load_profiles/raw/ercot_forecast'
for filename in os.listdir(dir):
    if '.143' not in filename:
        continue

    date_posted = pd.to_datetime(f"{filename.split('.')[3]} {filename.split('.')[4]}")

    f = os.path.join(dir, filename)
    df = pd.read_csv(f)
    df = df.loc[(
        (df['DeliveryDate'] == (date_posted + pd.Timedelta(days=1)).date().strftime('%m/%d/%Y'))
        & (df['InUseFlag'] == 'Y')
    )]

    forecast_profile_list_temp.append(df)

ercot_forecast_temp = (
    pd.concat(forecast_profile_list_temp, ignore_index=True)
    .drop_duplicates(keep='first')
)
ercot_forecast_temp['hour_ending'] = (
    ercot_forecast_temp.apply(
        axis=1,
        func=lambda x: (
            pd.to_datetime(x['DeliveryDate'])
            + pd.Timedelta(hours=int(x['HourEnding'].split(':')[0]))
        )
    )
    .astype(str)
)
ercot_forecast_temp.loc[ercot_forecast_temp.DSTFlag == 'Y', 'hour_ending'] = (
    ercot_forecast_temp.loc[ercot_forecast_temp.DSTFlag == 'Y', 'hour_ending']
    + ' DST'
)


forecast_profile_list = []
for year in range(2017, 2025):
    dst_start, dst_end = get_dst_timestamps(year)

    df = ercot_forecast_temp.loc[ercot_forecast_temp.DeliveryDate.str.contains(str(year))].copy()
    df['timestamp'] = (
        df['hour_ending']
        .apply(convert_to_utc_timestamp, dst_start=dst_start, dst_end=dst_end)
    )
    df = pd.melt(
        df,
        id_vars='timestamp',
        value_vars=ercot_weather_zone_subba_map.keys(),
        var_name='weather_zone'
    )
    df['subba'] = df['weather_zone'].map(ercot_weather_zone_subba_map)
    df = df.drop(columns='weather_zone')
    
    forecast_profile_list.append(df)

In [30]:
ercot_forecast = (
    pd.concat(forecast_profile_list, ignore_index=True)
    .drop_duplicates(keep='first')
)
ercot_forecast.to_csv(f"../data/iso_load_profiles/ercot_forecast.csv", index=False)